# AutoPipe — MVTec AD (anomalías industriales)

[MVTec AD](https://www.mvtec.com/company/research/datasets/mvtec-ad) es el
benchmark de referencia para detección de anomalías en imágenes industriales.
Cada categoría (p. ej. `bottle`, `cable`, `hazelnut`) trae:

```
<categoria>/
├── train/good/            # solo imágenes normales
└── test/
    ├── good/              # normales de prueba
    ├── <defecto_1>/       # anómalas
    └── <defecto_2>/ ...
```

Esa estructura no coincide con la que espera `ImageDataLoader`
(`normal/` + `anomaly/`), así que este ejemplo incluye un **adaptador** que la
convierte. Si no tienes MVTec AD descargado, el notebook genera un dataset
sintético con la misma estructura para que se ejecute de extremo a extremo.

> **Descarga real:** registra y descarga MVTec AD desde el enlace de arriba,
> descomprime una categoría y apunta `MVTEC_ROOT` a la carpeta `datasets/mvtec_ad`
> (carpeta local del repo, ignorada por git; ver `datasets/README.md`).

In [1]:
import shutil
import tempfile
from pathlib import Path

import numpy as np
from PIL import Image
from sklearn.ensemble import IsolationForest

from autopipe import (
    CLAHETransform,
    GaussianBlur,
    GridSearchEngine,
    ImageDataLoader,
    Normalize,
    ReportGenerator,
    Resize,
)

C:\Users\ricar\OneDrive - Estudiantes ITCR\2026 - I Semestre\Proyecto de Aplicación de Ingeniería\AutoPipe\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuración

In [2]:
MVTEC_ROOT = Path('datasets/mvtec_ad')   # raíz de tu descarga de MVTec AD
CATEGORY = 'bottle'                   # categoría a analizar
LIMIT_PER_CLASS = 40                 # imágenes por clase (acota el tiempo)

## 2. Adaptador de la estructura MVTec AD → `normal/` + `anomaly/`

El split `test/` ya mezcla normales (`good/`) y anómalas (resto de carpetas):
lo usamos para construir un dataset etiquetado para `ImageDataLoader`.

In [3]:
def build_normal_anomaly_dir(category_dir: Path, out_dir: Path, limit: int) -> Path:
    """Convierte <categoria>/test/ en una estructura normal/ + anomaly/."""
    test_dir = category_dir / 'test'
    (out_dir / 'normal').mkdir(parents=True, exist_ok=True)
    (out_dir / 'anomaly').mkdir(parents=True, exist_ok=True)
    for sub in sorted(p for p in test_dir.iterdir() if p.is_dir()):
        target = 'normal' if sub.name == 'good' else 'anomaly'
        for i, img in enumerate(sorted(sub.glob('*.png'))[:limit]):
            shutil.copy(img, out_dir / target / f'{sub.name}_{i:03d}.png')
    return out_dir


def synthesize_mvtec(category_dir: Path, n_good: int = 30, n_defect: int = 18) -> None:
    """Crea una estructura MVTec mínima sintética (demostración sin descarga).

    Es una simplificación pedagógica: las piezas 'good' son una textura gris
    uniforme y los defectos llevan un parche grande y brillante, claramente
    separable. Con MVTec AD real los defectos son sutiles y localizados.
    """
    rng = np.random.default_rng(0)
    test = category_dir / 'test'
    good = test / 'good'
    good.mkdir(parents=True, exist_ok=True)
    for i in range(n_good):
        base = rng.integers(110, 140)
        arr = np.clip(base + rng.normal(0, 6, (128, 128, 3)), 0, 255).astype(np.uint8)
        Image.fromarray(arr).save(good / f'{i:03d}.png')
    broken = test / 'broken'
    broken.mkdir(parents=True, exist_ok=True)
    for i in range(n_defect):
        arr = np.clip(125 + rng.normal(0, 6, (128, 128, 3)), 0, 255).astype(np.uint8)
        y, x = rng.integers(0, 64, size=2)
        arr[y:y + 64, x:x + 64] = rng.integers(210, 256, (64, 64, 3), dtype=np.uint8)
        Image.fromarray(arr).save(broken / f'{i:03d}.png')

In [4]:
category_dir = MVTEC_ROOT / CATEGORY
if not (category_dir / 'test').exists():
    print('MVTec AD no encontrado en', category_dir, '-> usando dataset sintético.')
    category_dir = Path(tempfile.mkdtemp()) / CATEGORY
    synthesize_mvtec(category_dir)
else:
    print('Usando MVTec AD real en', category_dir)

dataset_dir = build_normal_anomaly_dir(
    category_dir, Path(tempfile.mkdtemp()) / 'mvtec_ready', LIMIT_PER_CLASS
)
print('Dataset listo en', dataset_dir)

MVTec AD no encontrado en datasets\mvtec_ad\bottle -> usando dataset sintético.


Dataset listo en C:\Users\ricar\AppData\Local\Temp\tmp421hvid4\mvtec_ready


## 3. Cargar el dataset adaptado

In [5]:
images, labels = ImageDataLoader(dataset_path=dataset_dir).load()
print('Imágenes:', images.shape, '| normales:', int((labels == 0).sum()),
      '| anómalas:', int((labels == 1).sum()))

Imágenes: (48, 128, 128, 3) | normales: 30 | anómalas: 18


## 4. Espacio de búsqueda (preprocesamiento industrial)

Para imágenes industriales suele ayudar realzar el contraste local (CLAHE) y
reducir ruido. Buscamos sobre el tamaño del kernel de desenfoque y el límite
de recorte de CLAHE.

In [6]:
search_space = {
    'resize':    [Resize(width=64, height=64)],
    'clahe':     [CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),
                  CLAHETransform(clip_limit=4.0, tile_grid_size=(8, 8))],
    'blur':      [GaussianBlur(kernel_size=3), GaussianBlur(kernel_size=5)],
    'normalize': [Normalize(mean=0.5, std=0.5)],
}

## 5. Búsqueda en malla con `IsolationForest`

In [7]:
engine = GridSearchEngine(search_space, val_size=0.3, random_state=42)
best_pipeline, metrics, results = engine.run(
    images, labels, base_model=IsolationForest(random_state=0), metric='auc_roc',
)
print('Mejor pipeline:', best_pipeline)
for name, value in metrics.items():
    print(f'  {name:>20}: {value:.4f}')

Mejor pipeline: Pipeline([Resize(width=64, height=64), CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)), GaussianBlur(kernel_size=3), Normalize(mean=0.5, std=0.5)])
               auc_roc: 1.0000
                    f1: 1.0000
             precision: 1.0000
                recall: 1.0000
   false_positive_rate: 0.0000


## 6. Reporte con imágenes anómalas destacadas

Calculamos un puntaje de anomalía con el pipeline ganador para destacar las
imágenes de mayor score en el reporte.

In [8]:
features = np.vstack([best_pipeline.apply(img).ravel() for img in images])
scorer = IsolationForest(random_state=0).fit(features)
scores = -scorer.score_samples(features)   # mayor = más anómalo

# Sin output_dir la librería escribiría en reports/ junto al notebook; aquí
# usamos una subcarpeta explícita para no chocar con los demás ejemplos.
report = ReportGenerator(output_dir='reports/mvtec', top_anomalies=6).generate(
    results, best_pipeline, metrics, metric='auc_roc', images=images, scores=scores,
)
print('HTML:', report['html'])
print('PDF :', report['pdf'])
print('Config:', report['config'])

HTML: reports\mvtec\report.html
PDF : reports\mvtec\report.pdf
Config: reports\mvtec\winning_pipeline.json
